In [1]:
!pip install gradio

In [2]:
import sqlite3
import gradio as gr
import pandas as pd

In [3]:
conn = sqlite3.connect("expenses.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    amount REAL NOT NULL,
    category TEXT NOT NULL,
    date TEXT NOT NULL
)
""")

conn.commit()

print("Database created successfully!")

Database created successfully!


In [4]:
def add_expense(name, amount, category, date):
    if not name or not amount or not category or not date:
        return "Please fill all fields."

    try:
        amount = float(amount)
    except:
        return "Please enter a valid amount."

    cursor.execute(
        "INSERT INTO expenses (name, amount, category, date) VALUES (?, ?, ?, ?)",
        (name, amount, category, date)
    )

    conn.commit()

    return "Expense added successfully!"

In [5]:
def view_expenses():
    data = pd.read_sql_query(
        "SELECT * FROM expenses",
        conn
    )
    return data

In [6]:
def delete_expense(expense_id):
    try:
        expense_id = int(expense_id)
    except:
        return "Please enter a valid ID."

    cursor.execute(
        "DELETE FROM expenses WHERE id = ?",
        (expense_id,)
    )

    conn.commit()

    return "Expense deleted successfully!"

In [7]:
def total_expense():
    cursor.execute("SELECT SUM(amount) FROM expenses")
    total = cursor.fetchone()[0]

    if total is None:
        total = 0

    return f"Total Expense: ₹{total:.2f}"

In [8]:
with gr.Blocks(title="Expense Tracker") as app:

    gr.Markdown("# 💰 Expense Tracker")
    gr.Markdown("Simple personal expense management system")

    with gr.Tab("Add Expense"):
        name = gr.Textbox(label="Expense Name")
        amount = gr.Number(label="Amount")
        category = gr.Dropdown(
            ["Food", "Travel", "Shopping", "Education", "Bills", "Other"],
            label="Category"
        )
        date = gr.Textbox(label="Date (YYYY-MM-DD)")

        add_button = gr.Button("Add Expense")
        add_message = gr.Textbox(label="Message")

        add_button.click(
            add_expense,
            inputs=[name, amount, category, date],
            outputs=add_message
        )

    with gr.Tab("View Expenses"):
        view_button = gr.Button("View Expenses")
        expense_table = gr.Dataframe()

        view_button.click(
            view_expenses,
            outputs=expense_table
        )

    with gr.Tab("Delete Expense"):
        delete_id = gr.Number(label="Expense ID")
        delete_button = gr.Button("Delete Expense")
        delete_message = gr.Textbox(label="Message")

        delete_button.click(
            delete_expense,
            inputs=delete_id,
            outputs=delete_message
        )

    with gr.Tab("Total"):
        total_button = gr.Button("Calculate Total")
        total_output = gr.Textbox()

        total_button.click(
            total_expense,
            outputs=total_output
        )

In [9]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0a27f5533cc3c53da.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
